# SkySim PPO training on Google Colab (experimental)

This notebook trains an experimental PPO hover policy using the upstream SkySim **Godot headless Gymnasium server**. It is not connected to the browser's WebAssembly core, and it is not a model for the user's final physical quadcopter. Do not use it to control real hardware.

Colab is ephemeral: download the output ZIP before closing the session. GPU is optional.


In [ ]:
# Clone the project and pinned upstream source
!git clone https://github.com/samanramezani1377-hub/3d-fly-test.git
!git clone https://github.com/vishwagw/Sky-Sim-drone-simulator-3.0.git
%cd /content/Sky-Sim-drone-simulator-3.0
!git checkout a31864f999418531775980b879979ce4c57e791d
!python -m pip install -e 'python/[rl]'
%cd /content/3d-fly-test
!python -m pip install -r training/requirements.txt

## Start the simulator server

The upstream SkySim project requires its Godot extension to be built/available before the server can run. If your Colab runtime already has a compatible Godot 4 binary and the extension is built, start the server in the next cell. If not, stop here and follow the upstream build instructions first; a successful Python install alone is not enough to provide flight physics.


In [ ]:
import shutil, subprocess, time, os
godot = shutil.which('godot') or shutil.which('godot4')
if godot is None:
    raise RuntimeError('Godot executable not found. Install a compatible Godot 4 build and build the upstream extension before training.')
server_log = open('/content/skysim-agent-server.log', 'w')
server = subprocess.Popen([godot, '--headless', 'demo/agent_server.tscn', '--', '--port', '5557'], cwd='/content/Sky-Sim-drone-simulator-3.0', stdout=server_log, stderr=subprocess.STDOUT)
time.sleep(5)
if server.poll() is not None:
    print(open('/content/skysim-agent-server.log').read())
    raise RuntimeError('SkySim server exited. Check the log above; do not proceed until it stays running.')
print('SkySim headless server process started, PID:', server.pid)

## Train PPO

Start with a short smoke run. Once the server has been verified, increase `TIMESTEPS`. The run metadata explicitly records that this is a prototype backend.

In [ ]:
TIMESTEPS = 5000  # smoke test first; later try 200_000 or more
RUN_NAME = 'colab-prototype-hover'
!python training/train_ppo.py --port 5557 --timesteps {TIMESTEPS} --seed 42 --run-name {RUN_NAME}

In [ ]:
# Zip results for download from Colab
!cd training/runs && zip -r /content/{RUN_NAME}-results.zip {RUN_NAME}
from google.colab import files
files.download(f'/content/{RUN_NAME}-results.zip')

## Next step before aircraft-specific training

The next engineering task is to expose the same C++ `DroneCore` used by the browser WebAssembly build through a native binding and Gymnasium adapter, then test shared deterministic cases against the browser build. Until that exists, this notebook's rewards and flight behaviour belong only to the upstream Godot backend. Do not alter the repository's placeholder physical configuration or claim sim-to-real validity.